# VERTA: Semantic Segmentation Model Training on Google Colab (GPU)

This notebook trains a 3-class segmentation model (Background, Separator, Room Interior) using `segmentation_models_pytorch` with an ImageNet-pretrained ResNet34 encoder.

### Licensing Reminder
CubiCasa5K is hosted on Zenodo under the CC BY 4.0 license. Please verify compliance with the terms for your use case.

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi

## 2. Mount Google Drive
Mount Google Drive to persist checkpoints across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/verta_ml/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints will be saved to:', CHECKPOINT_DIR)

## 3. Unpack Training Bundle

In [ ]:
# Upload ml_bundle.zip to the root of /content or from Google Drive
!unzip -q /content/ml_bundle.zip -d /content/verta
%cd /content/verta

## 4. Install Training Dependencies

In [ ]:
!pip install -r ml/requirements-train.txt

## 5. Prepare Dataset
Run smoke data preparation first (200 samples), then prepare the full dataset.

In [ ]:
# Step 5a: Quick smoke prep (200 images)
!python ml/data/prepare_cubicasa.py --out-dir ml/data/dataset --limit 200

# Step 5b: Full dataset prep (uncomment when ready for full run)
# !python ml/data/prepare_cubicasa.py --out-dir ml/data/dataset

## 6. Local Pipeline Smoke Test (2 Epochs, GPU)
Quick verification to ensure the training loop, loss, metrics, and saving work properly.

In [ ]:
!python ml/train.py --data-dir ml/data/dataset --epochs 2 --batch-size 8 --out /content/smoke_models

## 7. Full Training Run with Checkpointing and Resume

In [ ]:
!python ml/train.py \
    --data-dir ml/data/dataset \
    --epochs 30 \
    --batch-size 16 \
    --lr 3e-4 \
    --out /content/drive/MyDrive/verta_ml/checkpoints \
    --resume /content/drive/MyDrive/verta_ml/checkpoints/latest_checkpoint.pth

## 8. Evaluation & Comparison with Classical Detector

In [ ]:
!python ml/evaluate.py \
    --model /content/drive/MyDrive/verta_ml/checkpoints/best_model.pth \
    --data-dir ml/data/dataset \
    --plans-dir plans \
    --out-dir ml/outputs/eval

## 9. Export to ONNX (Opset 17, Dynamic Spatial Dimensions)

In [ ]:
!python ml/export_onnx.py \
    --checkpoint /content/drive/MyDrive/verta_ml/checkpoints/best_model.pth \
    --out ml/models/verta_seg.onnx

## 10. Download Model Files for Local Deployment

In [ ]:
from google.colab import files
files.download('ml/models/verta_seg.onnx')
files.download('ml/models/model_card.json')